# 🚀 SimCSE: Unsupervised Pipeline, Ablations & Hugging Face Publishing
### **Google Colab Edition (Optimized for NVIDIA L4 GPU & High-RAM)**

**Author / Team Lead:** Russel Ku (Russ)  
**Team Responsibilities:** Unsupervised SimCSE pipeline, Unsupervised Ablation (*Same Dropout Mask*), Benchmark Evaluation, Hugging Face Hub Export & Verification.  
**Hardware Target:** NVIDIA L4 (24GB VRAM) / High-RAM Environment (53GB System RAM).  
**Paper Reference:** *SimCSE: Simple Contrastive Learning of Sentence Embeddings* (Gao et al., EMNLP 2021)  

---

## 📋 Workflow Outline
1. **⚡ Environment Setup & L4 Hardware Check:** Clone repository, install dependencies, enable CUDA & Tensor Cores.
2. **📊 Dataset Loading & Verification:** Process 100k SNLI dataset (~165k unique sentences) and STS-Benchmark (Dev/Test).
3. **🧪 Baseline Verification:** Evaluate Raw `bert-base-uncased` and `SBERT-2019` to verify STS-B harness.
4. **🎯 Unsupervised SimCSE Training:** Train `bert-base-uncased` with InfoNCE loss ($	au=0.05$) and independent dropout augmentation.
5. **🔬 Unsupervised Ablation Study:** Train with `same_dropout_mask=True` to empirically prove the collapse of contrastive learning without dropout noise.
6. **📈 Alignment & Uniformity Metrics:** Calculate Wang & Isola (2020) representation metrics.
7. **🤗 Hugging Face Hub Export & Verification:** Convert model to `SentenceTransformers`, generate Model Card, and verify reloading from the Hub.

## 1. ⚙️ Environment Setup & Repository Clone

In [1]:
# 1. Clone repository (or pull latest updates)
import os
if not os.path.exists("Sentence-embedding-model"):
    !git clone https://github.com/RusselKu/Sentence-embedding-model.git
    %cd Sentence-embedding-model
else:
    %cd Sentence-embedding-model
    !git pull

# 2. Install required packages (Fast)
!pip install -q --upgrade transformers datasets sentence-transformers scikit-learn scipy tqdm huggingface_hub

# 3. Verify CUDA & L4 GPU
import torch
print("=" * 60)
print(f"[✓] PyTorch Version: {torch.__version__}")
print(f"[✓] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[✓] Device Name: {torch.cuda.get_device_name(0)}")
    print(f"[✓] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"[✓] CUDA Capability: {torch.cuda.get_device_capability(0)}")
    # Enable TF32 for NVIDIA Ada Lovelace / L4 Tensor Cores speedup
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("[✓] TF32 Tensor Core Acceleration: ENABLED")
print("=" * 60)

Cloning into 'Sentence-embedding-model'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 59 (delta 20), reused 55 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 3.86 MiB | 23.14 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/Sentence-embedding-model
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 143.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 147.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 k

## 2. 📂 Ensure Dataset & Verify STS-Benchmark
Let's ensure `data/snli_train_100k.jsonl` is present and inspect the STS-B Dev/Test validation sets.

In [2]:
import os
import sys
sys.path.insert(0, os.getcwd())

from src.data import build_unsupervised_dataset, load_stsb_dataset

# Ensure data path
data_path = "data/snli_train_100k.jsonl"
if not os.path.exists(data_path) and os.path.exists("snli_train_100k.jsonl"):
    os.makedirs("data", exist_ok=True)
    !cp snli_train_100k.jsonl data/snli_train_100k.jsonl

# Load and verify unsupervised sentences
unique_sentences = build_unsupervised_dataset(data_path)
print(f"[✓] Extracted {len(unique_sentences):,} unique sentences from SNLI 100k.")

# Load STS-B
dev_records, test_records = load_stsb_dataset()
print(f"[✓] Loaded STS-B Benchmark: {len(dev_records)} Dev pairs, {len(test_records)} Test pairs.")
print(f"Sample STS-B pair: '{dev_records[0]['sentence1']}' <-> '{dev_records[0]['sentence2']}' (Score: {dev_records[0]['score']})")

[✓] Extracted 165,528 unique sentences from SNLI 100k.


README.md:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  471kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  142kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

[✓] Loaded STS-B Benchmark: 1500 Dev pairs, 1379 Test pairs.
Sample STS-B pair: 'A man with a hard hat is dancing.' <-> 'A man wearing a hard hat is dancing.' (Score: 1.0)


## 3. 🧪 Baseline Sanity Checks
Verify the STS-B evaluation harness against established reference scores:
- **Raw `bert-base-uncased` (Mean Pooling):** Expected Dev $\approx 59.31$ / Test $\approx 47.29$
- **SBERT-2019 (`bert-base-nli-mean-tokens`):** Expected Dev $\approx 80.77$ / Test $\approx 76.98$

In [3]:
!python scripts/run_sanity_checks.py

SIMCSE ENVIRONMENT & BASELINE SANITY CHECK
CUDA Available: True
GPU Device: NVIDIA L4

>>> 1. Evaluating Raw bert-base-uncased Baseline (Mean Pooling)...

--- Loading STS-B Dataset ---
Loaded 1500 dev pairs and 1379 test pairs.
Evaluating Model: bert-base-uncased (Type: raw_bert, Pooling: mean, Device: cuda)







[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different

## 4. 🎯 Run 1: Unsupervised SimCSE Training (Default Baseline)
- **Backbone:** `bert-base-uncased` (110M params)
- **Batch Size:** $64$
- **Learning Rate:** $3\times 10^{-5}$
- **Epochs:** $1$
- **Temperature ($\tau$):** $0.05$
- **Pooling:** `cls` token with MLP projection head during training
- **Data Augmentation:** Independent Dropout ($p=0.1$)

In [4]:
!python scripts/train_unsupervised.py \
  --data_path data/snli_train_100k.jsonl \
  --lr 3e-5 \
  --batch_size 64 \
  --epochs 1 \
  --temperature 0.05 \
  --pooling cls \
  --seed 42 \
  --run_name unsup_simcse_default

Training on device: cuda

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loaded STS-B: 1500 Dev, 1379 Test pairs.
Loading training data from data/snli_train_100k.jsonl for mode: unsup
Built Unsupervised dataset with 165528 unique sentences.

--- Starting Training: unsup_simcse_default ---
Total Steps: 2586

## 5. 🔬 Run 2: Unsupervised Ablation Study (*Same Dropout Mask*)

**Ablation Hypothesis:**
If we pass the sentence through the encoder twice with the **exact same dropout mask** (`same_dropout_mask=True`), both views become identical ($z_1 = z_2$). Without representation variance, the InfoNCE loss collapses towards trivial solutions and semantic quality drops.

In [5]:
!python scripts/run_ablations.py \
  --ablation unsup \
  --data_path data/snli_train_100k.jsonl


RUNNING ABLATION 1: Unsupervised SimCSE (Same Dropout Mask)
Training on device: cuda

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loaded STS-B: 1500 Dev, 1379 Test pairs.
Loading training data from data/snli_train_100k.jsonl for mode: unsup
Built Unsupervised dataset with 165528 unique sentences.

--- 

## 6. 📊 Comparative Metrics & Ablation Results Table

In [6]:
import json

with open("runs/unsup_simcse_default.json", "r") as f:
    unsup_res = json.load(f)
with open("runs/ablation_unsup_same_mask.json", "r") as f:
    ablation_res = json.load(f)

u_dev = unsup_res['results']['dev']['spearman']
u_test = unsup_res['results']['test']['spearman']
u_align = unsup_res['results']['test']['alignment']
u_unif = unsup_res['results']['test']['uniformity']

a_dev = ablation_res['results']['dev']['spearman']
a_test = ablation_res['results']['test']['spearman']
a_align = ablation_res['results']['test']['alignment']
a_unif = ablation_res['results']['test']['uniformity']

print("=" * 95)
print(f"{'Experiment Model':<35} | {'Dev Spearman':<14} | {'Test Spearman':<14} | {'Alignment (α=2)':<16} | {'Uniformity (t=2)'}")
print("=" * 95)
print(f"{'Unsup Default (Independent Dropout)':<35} | {u_dev:<14.2f} | {u_test:<14.2f} | {u_align:<16.4f} | {u_unif:.4f}")
print(f"{'Unsup Ablation (Same Dropout Mask)':<35} | {a_dev:<14.2f} | {a_test:<14.2f} | {a_align:<16.4f} | {a_unif:.4f}")
print("-" * 95)
print(f"[Δ Impact]: Independent Dropout yields +{u_dev - a_dev:.2f} Dev and +{u_test - a_test:.2f} Test Spearman points.")
print("=" * 95)

Experiment Model                    | Dev Spearman   | Test Spearman  | Alignment (α=2)  | Uniformity (t=2)
Unsup Default (Independent Dropout) | 76.32          | 67.32          | 0.6090           | -2.8900
Unsup Ablation (Same Dropout Mask)  | 55.39          | 47.78          | 0.7582           | -2.9761
-----------------------------------------------------------------------------------------------
[Δ Impact]: Independent Dropout yields +20.93 Dev and +19.54 Test Spearman points.


## 7. 🤗 Export & Publish to Hugging Face Hub (with Post-Upload Verification)

This step:
1. Converts the best PyTorch checkpoint into a native `SentenceTransformers` model.
2. Auto-generates a rich **Model Card (`README.md`)** with STS-B evaluation metrics and pipeline tags.
3. Uploads the model to Hugging Face Hub.
4. **Verification Step:** Reloads the model live from the Hub (`SentenceTransformer('<your-repo-id>')`) and validates STS-B Test score match.

In [7]:
from huggingface_hub import notebook_login

# 1. Login to Hugging Face (Paste your write token)
notebook_login()

In [19]:
%cd /content/Sentence-embedding-model
!git checkout origin/main -- src/publish.py


/content/Sentence-embedding-model


In [24]:
HF_TOKEN = "hf_your_token_here"
HF_REPO_ID = "RusselKuAguilar/simcse-bert-uncased-unsup"

!PYTHONPATH=. python src/publish.py \
  --checkpoint_dir checkpoints/unsup_simcse_default/best_checkpoint \
  --repo_id {HF_REPO_ID} \
  --run_json runs/unsup_simcse_default.json \
  --token {HF_TOKEN} \
  --pooling cls


/content/Sentence-embedding-model/src/publish.py:18: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers.models import Transformer, Pooling
Converting SimCSE checkpoint from checkpoints/unsup_simcse_default/best_checkpoint -> exported_model

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.Laye

## 8. 🧪 Live Test: Semantic Similarity Demo

In [25]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load your published model
model = SentenceTransformer(HF_REPO_ID)

sentences = [
    "A dog is running across the grass.",
    "A puppy is playing on the lawn outside.",
    "A chef is preparing a delicious Italian meal in the kitchen."
]

embeddings = model.encode(sentences, normalize_embeddings=True)
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:")
print(f"- 'Dog running' vs 'Puppy playing': {sim_matrix[0, 1]:.4f} (High semantic match)")
print(f"- 'Dog running' vs 'Chef cooking':   {sim_matrix[0, 2]:.4f} (Low semantic match)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cosine Similarity Matrix:
- 'Dog running' vs 'Puppy playing': 0.5833 (High semantic match)
- 'Dog running' vs 'Chef cooking':   0.1506 (Low semantic match)


## 9. 💾 Push Artifacts & Sync Run Logs back to GitHub

In [26]:
# Check generated run logs and artifacts
!ls -la runs/
!ls -la checkpoints/

total 28
drwxr-xr-x  2 root root 4096 Oct  2 16:15 .
drwxr-xr-x 10 root root 4096 Oct  2 16:30 ..
-rw-r--r--  1 root root  978 Oct  2 16:15 ablation_unsup_same_mask.json
-rw-r--r--  1 root root  284 Oct  2 15:51 baseline_raw_bert.json
-rw-r--r--  1 root root  311 Oct  2 15:51 baseline_sbert_2019.json
-rw-r--r--  1 root root 2116 Oct  2 16:15 run_history.json
-rw-r--r--  1 root root  968 Oct  2 16:06 unsup_simcse_default.json
total 16
drwxr-xr-x  4 root root 4096 Oct  2 16:06 .
drwxr-xr-x 10 root root 4096 Oct  2 16:30 ..
drwxr-xr-x  3 root root 4096 Oct  2 16:07 ablation_unsup_same_mask
drwxr-xr-x  3 root root 4096 Oct  2 15:53 unsup_simcse_default
